# Paper portfolio analysis

Uses generated synthetic prices for software research only; these are not actual market data.


In [ ]:
from pathlib import Path
import sys
repo = Path.cwd()
while not (repo / "backend").is_dir() and repo != repo.parent:
    repo = repo.parent
sys.path.insert(0, str(repo / "backend"))

from datetime import UTC, datetime, timedelta
from trading.broker.paper_broker import PaperBroker
from trading.performance.analytics import PerformanceAnalytics
from trading.portfolio.manager import PortfolioManager
broker = PaperBroker(initial_cash=1_000_000, fee_rate=0.0005, slippage_bps=5)
baseline_time = datetime(2024, 1, 1, tzinfo=UTC)
initial_equity = broker.equity
for symbol, quantity, price in (("DEMO_A", 10, 100), ("DEMO_B", 5, 200)):
    order = broker.create_order(symbol=symbol, side="BUY", quantity=quantity)
    broker.submit_order(order, market_price=price, timestamp=baseline_time)
    broker.fill_order(order.id, price=price, timestamp=baseline_time)
for symbol, opn, high, low, close in (("DEMO_A", 100, 112, 99, 110),
                                      ("DEMO_B", 200, 201, 188, 190)):
    broker.process_market_bar(symbol=symbol, open_price=opn, high=high, low=low,
                              close=close, timestamp=baseline_time + timedelta(days=1))
portfolio = PortfolioManager.from_broker(broker)
portfolio.snapshot()


In [ ]:
history = [
    {"captured_at": baseline_time, "equity": initial_equity, "exposure": 0.0},
    {"captured_at": baseline_time + timedelta(days=1), "equity": broker.equity,
     "exposure": broker.current_exposure},
]
PerformanceAnalytics.from_paper_broker(broker, history).snapshot()


This local simulation does not connect to an external broker or submit external orders.
